# GitSkills full-data profiling (Google Colab)

Runs Steps 3 to 6 of the profiling on the FULL GitSkills dataset (3.8M skill files, 282k repos).

**How to run:** Runtime > Run all. A normal (free) CPU runtime is fine. Keep the tab open.
The data is downloaded to Colab's disk, not your laptop (about 13 GB as Parquet).
Queries run with DuckDB straight on the Parquet files, so nothing has to fit in RAM.
Results are saved to `/content/results` and zipped at the end (and copied to Google Drive if you allow it).

**Decided before looking at any full-data result** (so the main comparison is not cherry-picked):
- A "collection repo" is a repo holding at least `MAIN_T` skill files (default 100). `SENS_T` are sensitivity checks.
- "Widely spread" means the same content sits in at least `WIDE` repos (default 10).
- The main comparison is: skills that bundle scripts vs skills that do not, among nested skills (not root-level `SKILL.md`),
  counting spread only across non-collection repos. Everything else is secondary and labelled as such.

If a cell fails, copy the error text and send it over.

In [ ]:
# ==== PARAMETERS (edit here) ====
REPO_ID = "mvaccargiu/gitskills"
WORKDIR = "/content/gitskills_data"   # where the Parquet files are downloaded (Colab disk)
LOCAL_DATA_DIR = None                 # set to a folder with the Parquet files to skip the download
OUT_DIR = "/content/results"          # results, plots and CSVs are written here
MAIN_T = 100                          # main "collection repo" threshold (skill files per repo)
SENS_T = [20, 1000]                   # sensitivity thresholds
WIDE = 10                             # "widely spread" = same content in >= WIDE repos
SAMPLE_MOD = 40                       # text analyses use a deterministic 1/SAMPLE_MOD sample of distinct skills
LANG_N = 20000                        # how many of those get language detection (slow part)
MEM_GB = 8                            # DuckDB memory limit

In [ ]:
%pip install -q duckdb huggingface_hub langdetect

import os, re, glob, shutil, collections, math
from urllib.parse import urlparse
import numpy as np, pandas as pd, duckdb
import matplotlib.pyplot as plt
from scipy import stats

os.makedirs(OUT_DIR + "/plots", exist_ok=True)
LOG = open(OUT_DIR + "/results.txt", "a")

def say(*a):
    s = " ".join(str(x) for x in a)
    print(s); LOG.write(s + "\n"); LOG.flush()

def show(title, df):
    say("\n### " + title)
    say(df.to_string(index=False))
    df.to_csv(OUT_DIR + "/" + re.sub(r"\W+", "_", title.lower()).strip("_") + ".csv", index=False)

def savefig(fig, name):
    fig.tight_layout(); fig.savefig(OUT_DIR + "/plots/" + name, dpi=150); plt.show(); plt.close(fig)

In [ ]:
# Optional: save results to your Google Drive (a pop-up will ask for permission). Skip with Cancel, nothing breaks.
DRIVE_OUT = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUT = "/content/drive/MyDrive/gitskills_results"
    os.makedirs(DRIVE_OUT, exist_ok=True)
except Exception as e:
    print("Drive not mounted:", e)

In [ ]:
# Download the Parquet mirror from Hugging Face to Colab's disk
du = shutil.disk_usage("/content" if os.path.exists("/content") else ".")
say("free disk (GB):", round(du.free / 1e9, 1), "(need about 25)")
if LOCAL_DATA_DIR is None:
    from huggingface_hub import snapshot_download
    LOCAL_DATA_DIR = snapshot_download(repo_id=REPO_ID, repo_type="dataset", local_dir=WORKDIR,
                                       allow_patterns=["*.parquet"])
files = sorted(glob.glob(LOCAL_DATA_DIR + "/**/*.parquet", recursive=True))
say(len(files), "parquet files,", round(sum(os.path.getsize(f) for f in files) / 1e9, 1), "GB")
for f in files[:60]:
    say("  ", os.path.relpath(f, LOCAL_DATA_DIR), round(os.path.getsize(f) / 1e6), "MB")

In [ ]:
# Connect DuckDB, find the files of each table, create views, sanity-check against the paper
con = duckdb.connect()
con.execute(f"SET memory_limit='{MEM_GB}GB'")
os.makedirs("/tmp/duck_tmp", exist_ok=True)
con.execute("SET temp_directory='/tmp/duck_tmp'")
con.execute("SET preserve_insertion_order=false")

def table_files(name):
    out = []
    for f in files:
        parts = os.path.relpath(f, LOCAL_DATA_DIR).replace(os.sep, "/").split("/")
        if name in parts[:-1] or parts[-1].startswith(name):
            out.append(f)
    return out

for t in ["artifacts", "artifact_siblings", "repos", "mining_runs"]:
    fs = table_files(t)
    assert fs, f"No Parquet files found for table '{t}'. Send the file list printed above."
    con.execute(f"CREATE OR REPLACE VIEW {t}_raw AS SELECT * FROM read_parquet({fs!r}, union_by_name=true)")

expected = {"artifacts": 3797117, "artifact_siblings": 7264865, "repos": 282200, "mining_runs": 7}
for t, e in expected.items():
    n = con.execute(f"SELECT COUNT(*) FROM {t}_raw").fetchone()[0]
    say(f"{t}: {n:,} rows, expected {e:,} ->", "OK" if n == e else "MISMATCH (fine for the sample, otherwise stop and check)")
for t in ["artifacts", "artifact_siblings", "repos"]:
    say(t, "columns:", [r[0] for r in con.execute(f"DESCRIBE {t}_raw").fetchall()])

# Standardised views. Text columns (content) are not included, so they are never read unless asked for.
con.execute("""CREATE OR REPLACE VIEW art AS
  SELECT repo_full_name, path, filename, location_class, file_sha, name, description,
         CAST(dedup_primary AS INTEGER) AS dedup_primary, body_chars,
         CAST(frontmatter_valid AS INTEGER) AS frontmatter_valid,
         CAST(has_scripts AS INTEGER) AS has_scripts, CAST(has_references AS INTEGER) AS has_references,
         CAST(composition_truncated AS INTEGER) AS composition_truncated,
         CAST(history_fetched AS INTEGER) AS history_fetched,
         sibling_count, first_commit_at, commit_count, first_commit_author, last_commit_author
  FROM artifacts_raw""")
con.execute("""CREATE OR REPLACE VIEW sib AS
  SELECT repo_full_name, artifact_path, entry_name, entry_type, entry_size FROM artifact_siblings_raw""")
con.execute("""CREATE OR REPLACE VIEW repo_meta AS
  SELECT full_name AS repo_full_name, owner, stars, forks, language, license, created_at, pushed_at,
         CAST(metadata_fetched AS INTEGER) AS metadata_fetched FROM repos_raw""")

In [ ]:
say("\n===== STEP 3: SIZE METRICS (full data) =====")
show("headline counts", con.execute("""
  SELECT COUNT(*)::BIGINT AS occurrences, SUM(dedup_primary)::BIGINT AS distinct_contents,
         COUNT(DISTINCT repo_full_name)::BIGINT AS repos_with_skills FROM art""").df())
show("repos table", con.execute("""
  SELECT COUNT(*)::BIGINT AS repos, COUNT(DISTINCT owner)::BIGINT AS owners,
         SUM(metadata_fetched)::BIGINT AS with_metadata FROM repo_meta""").df())
show("location class, all occurrences", con.execute("""
  SELECT location_class, COUNT(*)::BIGINT AS n, ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (), 1) AS pct
  FROM art GROUP BY 1 ORDER BY n DESC""").df())
say("(Note: the representative of each content prefers .claude/skills/, so location of distinct skills is biased. Use the occurrence view above.)")
show("front matter validity, distinct skills", con.execute("""
  SELECT frontmatter_valid, COUNT(*)::BIGINT AS n, ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (), 1) AS pct
  FROM art WHERE dedup_primary=1 GROUP BY 1 ORDER BY 1""").df())
show("bundling, distinct skills (root_level = SKILL.md at repo root)", con.execute("""
  SELECT (path='SKILL.md') AS root_level, COUNT(*)::BIGINT AS skills,
         SUM(has_scripts)::BIGINT AS with_scripts, ROUND(100.0*SUM(has_scripts)/COUNT(*), 1) AS pct_scripts,
         SUM(has_references)::BIGINT AS with_refs, ROUND(100.0*SUM(has_references)/COUNT(*), 1) AS pct_refs
  FROM art WHERE dedup_primary=1 AND has_scripts IS NOT NULL GROUP BY 1 ORDER BY 1""").df())
show("representatives with no folder listing (paper says 1,544)", con.execute("""
  SELECT COUNT(*)::BIGINT AS missing_folder_listing FROM art WHERE dedup_primary=1 AND has_scripts IS NULL""").df())
show("bundled files (siblings)", con.execute("""
  SELECT entry_type, COUNT(*)::BIGINT AS n, SUM(entry_size)::BIGINT AS bytes FROM sib GROUP BY 1""").df())
show("commit history and authors", con.execute("""
  SELECT (SELECT SUM(history_fetched)::BIGINT FROM art) AS rows_with_history,
         (SELECT COUNT(*)::BIGINT FROM (SELECT first_commit_author AS a FROM art
                                        UNION SELECT last_commit_author FROM art)
          WHERE a IS NOT NULL AND a <> '') AS distinct_authors_incl_bots""").df())

body = con.execute("SELECT body_chars FROM art WHERE dedup_primary=1 AND body_chars IS NOT NULL").df().body_chars
bsz = con.execute("SELECT entry_size FROM sib WHERE entry_type='file' AND entry_size IS NOT NULL").df().entry_size
def summ(name, s):
    return {"variable": name, "count": len(s), "mean": s.mean(), "median": s.median(), "std": s.std(),
            "variance": s.var(), "min": s.min(), "q25": s.quantile(.25), "q75": s.quantile(.75),
            "max": s.max(), "skew": s.skew(), "kurtosis": s.kurt()}
show("summary statistics", pd.DataFrame([summ("skill body_chars (distinct)", body),
                                         summ("bundled file entry_size (bytes)", bsz)]).round(2))

In [ ]:
say("\n===== STEP 4: DISTRIBUTIONS AND REUSE (full data) =====")
con.execute("""CREATE OR REPLACE TABLE spread AS
  SELECT file_sha, COUNT(*)::BIGINT AS copies, COUNT(DISTINCT repo_full_name)::BIGINT AS n_repos
  FROM art GROUP BY file_sha""")
con.execute("""CREATE OR REPLACE TABLE reps AS
  SELECT file_sha, repo_full_name AS rep_repo, path AS rep_path, filename, location_class,
         has_scripts, has_references, sibling_count, name, description, body_chars
  FROM art WHERE dedup_primary=1""")
con.execute("""CREATE OR REPLACE TABLE repo_size AS
  SELECT repo_full_name, COUNT(*)::BIGINT AS n_files FROM art GROUP BY 1""")

sp = con.execute("SELECT copies, n_repos FROM spread").df()
n_occ, n_dist = int(sp.copies.sum()), len(sp)
top1 = sp.copies.sort_values(ascending=False).head(int(n_dist * 0.01)).sum() / n_occ * 100
show("reuse concentration", pd.DataFrame([{
    "occurrences": n_occ, "distinct_contents": n_dist,
    "pct_occurrences_that_are_copies": round((1 - n_dist / n_occ) * 100, 1),
    "pct_contents_appearing_once": round((sp.copies == 1).mean() * 100, 1),
    "pct_contents_in_exactly_one_repo": round((sp.n_repos == 1).mean() * 100, 1),
    "max_copies": int(sp.copies.max()), "max_repos": int(sp.n_repos.max()),
    "share_of_occurrences_in_top_1pct_contents": round(top1, 1)}]))
buckets = pd.cut(sp.copies, [0, 1, 2, 9, 99, 10**9], labels=["1", "2", "3-9", "10-99", "100+"]).value_counts().sort_index()
show("distinct contents by number of copies", buckets.rename_axis("copies").reset_index(name="distinct_contents"))

vc = sp.copies.value_counts().sort_index()
fig, (a, b) = plt.subplots(1, 2, figsize=(11, 4))
a.loglog(vc.index, vc.values, "o", ms=3); a.set_xlabel("copies of the same content"); a.set_ylabel("number of distinct contents"); a.set_title("Copies per distinct content")
b.bar(buckets.index.astype(str), buckets.values); b.set_yscale("log"); b.set_xlabel("copies"); b.set_title("Bucketed (compare with the authors' plot)")
savefig(fig, "copies.png")

def hist_log(s, title, xlabel, fname):
    s = s[s > 0]
    bins = np.logspace(np.log10(s.min()), np.log10(s.max()), 60)
    fig, (a, b) = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={"width_ratios": [3, 1]})
    a.hist(s, bins=bins); a.set_xscale("log"); a.set_xlabel(xlabel); a.set_ylabel("count"); a.set_title(title)
    b.boxplot(np.log10(s), showfliers=False); b.set_ylabel("log10(" + xlabel + "), outliers not drawn"); b.set_xticks([])
    savefig(fig, fname)

hist_log(body, "Skill body length (distinct skills)", "body_chars", "body_size.png")
hist_log(bsz, "Bundled file size", "entry_size (bytes)", "bundled_size.png")
stars = con.execute("SELECT stars FROM repo_meta WHERE metadata_fetched=1 AND stars IS NOT NULL").df().stars
say("\nrepos with 0 stars (%):", round((stars == 0).mean() * 100, 1), "| median stars among starred:", stars[stars > 0].median())
hist_log(stars, "Stars per repo (repos with 1+ stars)", "stars", "stars.png")

In [ ]:
say("\n===== STEP 5: TRACEABILITY (full data) =====")
SCRIPT_EXT = {".py": "Python", ".sh": "Shell", ".bash": "Shell", ".js": "JavaScript", ".mjs": "JavaScript",
              ".cjs": "JavaScript", ".ts": "TypeScript", ".rb": "Ruby", ".go": "Go", ".rs": "Rust",
              ".ps1": "PowerShell", ".bat": "Batch", ".java": "Java", ".php": "PHP", ".pl": "Perl", ".lua": "Lua"}
con.register("ext_map", pd.DataFrame({"ext": list(SCRIPT_EXT), "plang": list(SCRIPT_EXT.values())}))

show("top 25 extensions of bundled files", con.execute(r"""
  SELECT COALESCE(NULLIF(lower(regexp_extract(entry_name, '(\.[^./]+)$', 1)), ''), '(none)') AS ext,
         COUNT(*)::BIGINT AS n FROM sib WHERE entry_type='file' GROUP BY 1 ORDER BY n DESC LIMIT 25""").df())
show("script files by language", con.execute(r"""
  SELECT m.plang, COUNT(*)::BIGINT AS files FROM sib s
  JOIN ext_map m ON lower(regexp_extract(s.entry_name, '(\.[^./]+)$', 1)) = m.ext
  WHERE s.entry_type='file' GROUP BY 1 ORDER BY files DESC""").df())
con.execute(r"""CREATE OR REPLACE TABLE script_skills AS
  SELECT s.repo_full_name, s.artifact_path, COUNT(*)::BIGINT AS n_script_files, COUNT(DISTINCT m.plang)::BIGINT AS n_langs
  FROM sib s JOIN ext_map m ON lower(regexp_extract(s.entry_name, '(\.[^./]+)$', 1)) = m.ext
  WHERE s.entry_type='file' GROUP BY 1, 2""")
show("number of script languages per skill (skills with 1+ script file)", con.execute("""
  SELECT n_langs, COUNT(*)::BIGINT AS skills FROM script_skills GROUP BY 1 ORDER BY 1""").df())
say("\n(Slow query: counts bundled files that have stored text.)")
show("bundled files with stored text", con.execute("""
  SELECT COUNT(*)::BIGINT AS bundled_files,
         SUM(CASE WHEN content IS NOT NULL THEN 1 ELSE 0 END)::BIGINT AS with_text
  FROM artifact_siblings_raw WHERE entry_type='file'""").df())

In [ ]:
# Text analyses on a deterministic 1/SAMPLE_MOD sample of distinct skills (full text is too big for plain Python)
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0
txt = con.execute(f"""SELECT file_sha, content FROM artifacts_raw
  WHERE CAST(dedup_primary AS INTEGER)=1 AND content IS NOT NULL AND hash(file_sha) % {SAMPLE_MOD} = 0""").df()
say(f"\ntext sample: {len(txt):,} distinct skills (1/{SAMPLE_MOD} by content hash)")

url_re = re.compile(r"https?://[^\s)>\]\"'`]+")
PLACE = {"example.com", "example.org", "example.net", "localhost", "127.0.0.1", "0.0.0.0", "...",
         "your-domain.com", "yourdomain.com", "domain.com", "your-server.com"}
def is_placeholder(h):
    h = h.split(":")[0].strip(".")
    return (h in PLACE or h.startswith("localhost") or h.endswith("example.com") or h.endswith(".example")
            or h.startswith("your") or "{" in h or "$" in h or "<" in h or h == "" or "." not in h)
n_raw = n_real = n_urls = 0
dom = collections.Counter()
for c in txt.content:
    urls = url_re.findall(c)
    hosts = set()
    for u in urls:
        try: hosts.add(urlparse(u).netloc.lower())
        except ValueError: pass
    real = {h for h in hosts if not is_placeholder(h)}
    n_raw += bool(urls); n_real += bool(real); n_urls += len(urls); dom.update(real)
show("URLs in skill text (text sample)", pd.DataFrame([{
    "skills": len(txt), "with_any_url": n_raw, "with_real_external_url": n_real,
    "pct_with_real_url": round(100 * n_real / max(len(txt), 1), 1), "total_urls": n_urls}]))
show("top 25 real domains (skills mentioning each)", pd.DataFrame(dom.most_common(25), columns=["domain", "skills"]))

def lang(t):
    t = re.sub(r"```.*?```", " ", t, flags=re.S)
    try: return detect(t[:2000])
    except Exception: return "unknown"
lt = txt.sort_values("file_sha").head(LANG_N).copy()
lt["lang"] = lt.content.map(lang)
lv = lt.lang.value_counts()
lv = pd.DataFrame({"lang": lv.index, "skills": lv.values, "pct": (100 * lv.values / len(lt)).round(1)})
show(f"natural language of skills (first {len(lt):,} of the text sample, noisy on short text)", lv.head(20))

vocab, lines = collections.Counter(), 0
for c in txt.content:
    vocab.update(re.findall(r"\w+", c.lower())); lines += c.count("\n") + 1
show("vocabulary (text sample only, grows with sample size)", pd.DataFrame([{
    "skills": len(txt), "unique_tokens": len(vocab), "total_tokens": sum(vocab.values()), "total_lines": lines}]))

In [ ]:
# Over time: first commit of the SKILL.md file (history sample only, not random)
h = con.execute("""SELECT substr(CAST(first_commit_at AS VARCHAR), 1, 7) AS ym, COUNT(*)::BIGINT AS n
  FROM art WHERE history_fetched=1 AND first_commit_at IS NOT NULL AND filename='SKILL.md'
  GROUP BY 1 ORDER BY 1""").df()
say("\nfirst commits before 2025-10 (before the format existed):", int(h[h.ym < "2025-10"].n.sum()), "of", int(h.n.sum()))
m = h[h.ym >= "2025-09"]
show("first commit per month (history sample, exact SKILL.md only)", m)
fig = plt.figure(figsize=(9, 4)); plt.bar(m.ym, m.n); plt.xticks(rotation=60)
plt.ylabel("skills (history sample)"); plt.title("First commit of SKILL.md files by month (July 2026 is partial)")
savefig(fig, "over_time.png")

In [ ]:
say("\n===== STEP 6: SCRIPTS VS HOW WIDELY SKILLS SPREAD (full data) =====")
show("has_scripts flag (rows) vs extension-based script files (ext_has_script)", con.execute("""
  SELECT a.has_scripts AS flag, (s.n_script_files IS NOT NULL) AS ext_has_script, COUNT(*)::BIGINT AS skills
  FROM art a LEFT JOIN script_skills s ON a.repo_full_name=s.repo_full_name AND a.path=s.artifact_path
  WHERE a.dedup_primary=1 GROUP BY 1, 2 ORDER BY 1, 2""").df())

def ptest(hit0, n0, hit1, n1):
    """0 = no scripts, 1 = scripts. Returns odds ratio (scripts vs none), 95% CI, p-value."""
    t = [[hit1, n1 - hit1], [hit0, n0 - hit0]]
    if min(min(t[0]), min(t[1])) == 0: return (np.nan, np.nan, np.nan, np.nan)
    p = stats.fisher_exact(t)[1] if min(min(t[0]), min(t[1])) < 1000 else stats.chi2_contingency(t)[1]
    OR = (hit1 / (n1 - hit1)) / (hit0 / (n0 - hit0))
    se = math.sqrt(1/hit1 + 1/(n1 - hit1) + 1/hit0 + 1/(n0 - hit0))
    return OR, OR * math.exp(-1.96 * se), OR * math.exp(1.96 * se), p

def spread_counts(T=None, nested=True, strict=False):
    excl = f"AND repo_full_name NOT IN (SELECT repo_full_name FROM repo_size WHERE n_files >= {T})" if T else ""
    occ_f = "AND filename='SKILL.md'" if strict else ""
    rep_f = ("AND r.rep_path <> 'SKILL.md' " if nested else "") + ("AND r.filename='SKILL.md'" if strict else "")
    df = con.execute(f"""
      WITH c AS (SELECT file_sha, COUNT(DISTINCT repo_full_name)::BIGINT AS n FROM art
                 WHERE 1=1 {excl} {occ_f} GROUP BY file_sha)
      SELECT r.has_scripts AS hs, COUNT(*)::BIGINT AS skills,
             COUNT(*) FILTER (WHERE c.n > 1)::BIGINT AS in_2plus,
             COUNT(*) FILTER (WHERE c.n >= {WIDE})::BIGINT AS in_wide
      FROM reps r JOIN c ON r.file_sha = c.file_sha
      WHERE r.has_scripts IS NOT NULL {rep_f} GROUP BY r.has_scripts ORDER BY r.has_scripts""").df().set_index("hs")
    return df.loc[0], df.loc[1]

settings = [("all repos, all skills (headline, naive)", None, False, False),
            ("all repos, nested skills only", None, True, False),
            (f"MAIN: exclude collection repos (>= {MAIN_T}), nested only", MAIN_T, True, False)]
settings += [(f"sensitivity: exclude repos >= {t}, nested only", t, True, False) for t in SENS_T]
settings += [(f"sensitivity: MAIN but exact basename SKILL.md only", MAIN_T, True, True)]
rows = []
for name, T, nested, strict in settings:
    z, o = spread_counts(T, nested, strict)
    r2 = ptest(z.in_2plus, z.skills, o.in_2plus, o.skills)
    rw = ptest(z.in_wide, z.skills, o.in_wide, o.skills)
    rows.append({"setting": name, "n_no_scripts": z.skills, "n_scripts": o.skills,
                 "pct_2plus_no": round(100 * z.in_2plus / z.skills, 2), "pct_2plus_scr": round(100 * o.in_2plus / o.skills, 2),
                 "OR_2plus": round(r2[0], 2), "CI_2plus": f"{r2[1]:.2f}-{r2[2]:.2f}", "p_2plus": f"{r2[3]:.1e}",
                 f"pct_{WIDE}plus_no": round(100 * z.in_wide / z.skills, 2), f"pct_{WIDE}plus_scr": round(100 * o.in_wide / o.skills, 2),
                 f"OR_{WIDE}plus": round(rw[0], 2), f"CI_{WIDE}plus": f"{rw[1]:.2f}-{rw[2]:.2f}", f"p_{WIDE}plus": f"{rw[3]:.1e}"})
show("scripts vs spread, all settings (OR > 1 means script skills are MORE likely to spread)", pd.DataFrame(rows))
say("Caution: skills cluster inside repos, so these p-values assume more independence than the data has. "
    "With millions of rows almost anything is 'significant', so read the odds ratios and the direction across settings.")

In [ ]:
# Which skills are widely spread and bundle scripts, and where do their copies live?
show(f"top 30 skills that bundle scripts and sit in >= {WIDE} repos", con.execute(f"""
  SELECT s.n_repos, s.copies, r.name, r.location_class, r.sibling_count, substr(r.description, 1, 70) AS descr
  FROM reps r JOIN spread s ON r.file_sha = s.file_sha
  WHERE r.has_scripts = 1 AND s.n_repos >= {WIDE} ORDER BY s.n_repos DESC LIMIT 30""").df())
show(f"copies of widely spread skills (>= {WIDE} repos) that sit in collection repos (>= {MAIN_T} files)", con.execute(f"""
  SELECT r.has_scripts, COUNT(*)::BIGINT AS copies,
         COUNT(*) FILTER (WHERE rs.n_files >= {MAIN_T})::BIGINT AS in_collection_repos,
         ROUND(100.0 * COUNT(*) FILTER (WHERE rs.n_files >= {MAIN_T}) / COUNT(*), 1) AS pct_in_collection_repos
  FROM art a JOIN spread s ON a.file_sha = s.file_sha JOIN reps r ON a.file_sha = r.file_sha
  JOIN repo_size rs ON a.repo_full_name = rs.repo_full_name
  WHERE s.n_repos >= {WIDE} AND r.has_scripts IS NOT NULL GROUP BY r.has_scripts ORDER BY r.has_scripts""").df())
show(f"repos holding the most widely spread script skills (top 20)", con.execute(f"""
  SELECT a.repo_full_name, COUNT(DISTINCT a.file_sha)::BIGINT AS n_wide_script_skills, rs.n_files AS skill_files_in_repo
  FROM art a JOIN spread s ON a.file_sha = s.file_sha JOIN reps r ON a.file_sha = r.file_sha
  JOIN repo_size rs ON a.repo_full_name = rs.repo_full_name
  WHERE s.n_repos >= {WIDE} AND r.has_scripts = 1 GROUP BY 1, 3 ORDER BY 2 DESC LIMIT 20""").df())
show("repo size: how many skill files do repos hold (full data, not thinned)", con.execute("""
  SELECT CASE WHEN n_files=1 THEN '1' WHEN n_files<=5 THEN '2-5' WHEN n_files<=20 THEN '6-20'
              WHEN n_files<=100 THEN '21-100' WHEN n_files<=1000 THEN '101-1000' ELSE '1001+' END AS skill_files_in_repo,
         COUNT(*)::BIGINT AS repos, SUM(n_files)::BIGINT AS occurrences,
         ROUND(100.0 * SUM(n_files) / SUM(SUM(n_files)) OVER (), 1) AS pct_of_occurrences
  FROM repo_size GROUP BY 1 ORDER BY MIN(n_files)""").df())

In [ ]:
# Package results
LOG.flush()
shutil.make_archive("/content/gitskills_results", "zip", OUT_DIR)
if DRIVE_OUT:
    shutil.copytree(OUT_DIR, DRIVE_OUT, dirs_exist_ok=True)
    say("copied to Drive:", DRIVE_OUT)
try:
    from google.colab import files as colab_files
    colab_files.download("/content/gitskills_results.zip")
except Exception as e:
    print("zip is at /content/gitskills_results.zip", e)